# 07 Generate GOBM SOCAT-Mask Weights and DII Intermediates From Romy

Purpose: generate the GOBM feature-weight and DII intermediates within the SOCAT mask that were provided by Romy/Romina and used by comparison plotting scripts.

Inputs:
- GOBM `.pkl` files under `gs://leap-persistent/vacquaviva/GOBMs/` or an equivalent local directory

Outputs:
- `FromRomy/Finalweights_sampled1_seed10_<model>.txt` through seed 14
- `FromRomy/Finalimbs_sampled1_seed10_<model>.txt` through seed 14

where `<model>` is `ETHZ`, `FESOM`, `IPSL`, `MRI`, and `NorESM`.

Estimated runtime: very slow. Expect multi-hour runtimes per random seed in a typical analysis environment.

Notes:
- This notebook is a provenance workflow for generating the GOBM SOCAT-mask weight and DII files.
- This optional regeneration step requires large upstream GOBM `.pkl` files and a compatible Python environment.
- The output directory is kept as `intermediates/FromRomy/` to preserve the external provenance boundary.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from dadapy.feature_weighting import FeatureWeighting
from sklearn.preprocessing import StandardScaler

# Use the GCS root by default. Set this to a local folder if the pkl files have been downloaded.
GOBM_ROOT = "gs://leap-persistent/vacquaviva/GOBMs"
OUTPUT_DIR = Path("../intermediates/FromRomy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 16000
SEEDS = [10, 11, 12, 13, 14]
N_EPOCHS = 80
TIME_MIN = "2020-01-01"


## Model Files and Feature Definitions


In [ ]:
MODEL_FILES = {
    "ETHZ": "CESM_ETHZ/MLinput_CESM_ETHZ_mon_1x1_197001_202212.pkl",
    "FESOM": "FESOM2_REcoM/MLinput_FESOM2_REcoM_mon_1x1_197001_202212.pkl",
    "IPSL": "IPSL/MLinput_IPSL_mon_1x1_197001_202212.pkl",
    "MRI": "MRI_ESM2_2/MLinput_MRI_ESM2_2_mon_1x1_197001_202212.pkl",
    "NorESM": "NorESM/MLinput_NorESM_mon_1x1_197001_202212.pkl",
}

FEATURES = [
    "sst",
    "sst_anom",
    "sss",
    "sss_anom",
    "chl_log",
    "chl_log_anom",
    "mld_log",
    "xco2",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"
SOCAT_MASK = "socat_mask"


## Helper Functions


In [ ]:
def model_path(model):
    root = str(GOBM_ROOT).rstrip("/")
    return f"{root}/{MODEL_FILES[model]}"

def load_model_socatmask_scaled(model):
    print(f"Loading {model}")
    frame = pd.read_pickle(model_path(model))
    frame[TARGET] = frame["sfco2"] - frame["xco2"]
    frame = frame[FEATURES + [SOCAT_MASK, TARGET]]
    frame = frame[frame[SOCAT_MASK] == 1.0]
    frame = frame[frame.index.get_level_values("time") >= TIME_MIN]

    scaled = StandardScaler().fit_transform(frame.loc[:, FEATURES + [TARGET]])
    scaled = pd.DataFrame(scaled, columns=FEATURES + [TARGET])
    scaled = scaled.drop_duplicates(subset=[TARGET]).dropna()
    print(model, "scaled rows:", scaled.shape[0])
    return scaled

def run_model_seed(model, scaled, seed):
    print(f"Starting {model}, seed {seed}")
    sample = scaled.sample(n=N_SAMPLE, random_state=seed)

    target = FeatureWeighting(sample[[TARGET]].to_numpy(), verbose=True)
    inputs = FeatureWeighting(sample[FEATURES].to_numpy(), verbose=True)

    final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
        target_data=target,
        initial_weights=None,
        n_epochs=N_EPOCHS,
        learning_rate=None,
        decaying_lr="cos",
    )

    weights_path = OUTPUT_DIR / f"Finalweights_sampled1_seed{seed}_{model}.txt"
    imbs_path = OUTPUT_DIR / f"Finalimbs_sampled1_seed{seed}_{model}.txt"
    np.savetxt(weights_path, final_weights)
    np.savetxt(imbs_path, final_imbs)
    print(f"Saved {weights_path}")
    print(f"Saved {imbs_path}")


## Generate Outputs

Very slow cell. It loops over all five models and seeds 10-14.

If a run is interrupted, you can restrict `MODELS_TO_RUN` or `SEEDS_TO_RUN` and rerun only the missing combinations.


In [ ]:
MODELS_TO_RUN = list(MODEL_FILES)
SEEDS_TO_RUN = SEEDS

for model in MODELS_TO_RUN:
    scaled_model = load_model_socatmask_scaled(model)
    for seed in SEEDS_TO_RUN:
        run_model_seed(model, scaled_model, seed)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = []
for model in MODEL_FILES:
    for seed in SEEDS:
        expected_outputs.extend([
            f"Finalweights_sampled1_seed{seed}_{model}.txt",
            f"Finalimbs_sampled1_seed{seed}_{model}.txt",
        ])

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
